In [1]:
import tifffile as tiff
from pathlib import Path
import pandas as pd

import torch
import torchvision.transforms.functional as TF

Loading the data for preprocessing

In [2]:
TrainingPath = Path("..") / "Training data"
TrainingImgPath = TrainingPath / "train_composite"
ValidationPath = Path("..") / "Evaluation data"
ValidationImgPath = ValidationPath / "evaluation_composite"
SavePath = Path("..") / "Processed data"

train_tabular = pd.read_csv(TrainingPath / "train_tabular.csv")
eval_tabular = pd.read_csv(ValidationPath / "evaluation_tabular_no_target.csv")
print(f"Tabular shape: {train_tabular.shape}")

Tabular shape: (1024, 23)


Deal with missing data

In [3]:
#Finding missing data
def find_missing_data(data_tabular, imgs_path):
    """Find missing data in the tabular data"""
    missing_data = data_tabular.isnull().sum()
    missing_data = missing_data[missing_data > 0]
    print("Columns with missing data:")
    print(missing_data)

    sentinel_files = set()
    viirs_files = set()

    for idx, row in data_tabular.iterrows():
        sentinel_files.add(row['sentinel2_tiff_file_name'])
        viirs_files.add(row['viirs_tiff_file_name'])

    print(f"Total referenced sentinel files: {len(sentinel_files)}")
    print(f"Total referenced viirs files: {len(viirs_files)}")

    for file in imgs_path.iterdir():
        if file.name in sentinel_files:
            sentinel_files.remove(file.name)
        elif file.name in viirs_files:
            viirs_files.remove(file.name)
        else:
            print(f"Unreferenced file: {file.name}")

    print(f"Missing sentinel files: {len(sentinel_files)}")
    print(f"Missing viirs files: {len(viirs_files)}")

    print("Data integrity check complete.")
    return missing_data

missing_train = find_missing_data(train_tabular, TrainingImgPath)
missing_eval = find_missing_data(eval_tabular, ValidationImgPath)

Columns with missing data:
tropical_cyclone_wind_risk    4
dtype: int64
Total referenced sentinel files: 1024
Total referenced viirs files: 1024
Missing sentinel files: 0
Missing viirs files: 0
Data integrity check complete.
Columns with missing data:
Series([], dtype: int64)
Total referenced sentinel files: 1024
Total referenced viirs files: 1024
Missing sentinel files: 0
Missing viirs files: 0
Data integrity check complete.


In [4]:
#Print rows with missing data
def print_missing_data(missing_data):
    if not missing_data.empty:
        print("Rows with missing data:")
        #Row will consist of geolocation_name, quarter_label, year, and tropical_cyclone_wind_risk
        rows_with_missing = train_tabular[train_tabular.isnull().any(axis=1)][['geolocation_name', 'quarter_label', 'year', 'tropical_cyclone_wind_risk']]
        print(rows_with_missing)

        #Make a set of unique geolocation names with missing data
        unique_geolocations_with_missing = set(rows_with_missing['geolocation_name'].unique())

        #Other rows with the same geolocation_name
        for geolocation in unique_geolocations_with_missing:
            similar_rows = train_tabular[train_tabular['geolocation_name'] == geolocation][['geolocation_name', 'quarter_label', 'year', 'tropical_cyclone_wind_risk']]
            similar_rows.sort_values(by=['year', 'quarter_label'], inplace=True)
            print(f"Other rows with geolocation_name {geolocation}:")
            print(similar_rows)
            #Get the most common categorical value tropical_cyclone_wind_risk value for these rows
            most_common_value = similar_rows['tropical_cyclone_wind_risk'].mode()[0]
            print(f"Most common tropical_cyclone_wind_risk value for {geolocation}: {most_common_value}")
            #Fill missing tropical_cyclone_wind_risk values with the most common value
            train_tabular.loc[(train_tabular['geolocation_name'] == geolocation) & (train_tabular['tropical_cyclone_wind_risk'].isnull()), 'tropical_cyclone_wind_risk'] = most_common_value
    else: print("No missing values found")

print_missing_data(missing_train)
print_missing_data(missing_eval)

Rows with missing data:
    geolocation_name quarter_label  year tropical_cyclone_wind_risk
239      25000 Shiga       2022-Q4  2022                        NaN
382      25000 Shiga       2022-Q1  2022                        NaN
460      25000 Shiga       2022-Q3  2022                        NaN
743      25000 Shiga       2022-Q2  2022                        NaN
Other rows with geolocation_name 25000 Shiga:
    geolocation_name quarter_label  year tropical_cyclone_wind_risk
101      25000 Shiga       2019-Q1  2019                   Moderate
915      25000 Shiga       2019-Q3  2019                   Moderate
188      25000 Shiga       2019-Q4  2019                   Moderate
81       25000 Shiga       2020-Q2  2020                        Low
226      25000 Shiga       2020-Q3  2020                        Low
520      25000 Shiga       2020-Q4  2020                        Low
106      25000 Shiga       2021-Q1  2021                   Moderate
547      25000 Shiga       2021-Q2  2021      

Convert categorical columns to numeric

In [5]:
def convert_to_nummeric(df):
    numeric_columns = df.select_dtypes(include=['number']).columns
    print(f"Numeric columns: {numeric_columns.tolist()}")
    nonnumeric_columns = df.select_dtypes(exclude=['number']).columns
    print(f"Nonnumeric columns before processing: {nonnumeric_columns.tolist()}")

    #Convert geolocation_id into numeric format
    location_mapping = {loc: idx for idx, loc in enumerate(df['geolocation_name'].unique())}
    df['geolocation_name'] = df['geolocation_name'].map(location_mapping).astype('Int64')

    #Convert quarter_labels into a numeric format like 2020.25 for 2020-Q1, 2019.5 for 2019-Q2, etc.
    quarter_mapping = {}
    for quater in df['quarter_label'].unique():
        year, quarter = quater.split('-Q')
        quarter_mapping[quater] = int(year) + (int(quarter) - 1) * 0.25

    df['quarter_label'] = df['quarter_label'].map(quarter_mapping)
    #print(f"Unique quarter labels: {df['quarter_label'].nunique()}")
    #print("Columns after processing:")
    #print(df['quarter_label'].head())

    #Convert yes/no columns to 1/0
    yes_no_columns = ['developed_country', 'landlocked', 'access_to_airport', 'access_to_port', 'access_to_highway', 'access_to_railway', 'flood_risk_class']
    for col in yes_no_columns:
        df[col] = df[col].map({'Yes': 1, 'No': 0}).astype('Int64')
        #print(f"Converted column {col} to numeric.")
        #print(df[col].head())

    #Convert country to "japan" = 0 and "philipines" = 1
    df["country"] = df["country"].map({'Philippines': 0, 'Japan': 1}).astype('Int64')


    #Convert region_economic_classification to numeric codes
    economic_map = {
        'Low income': 0,
        'Lower-middle income': 1,
        'Upper-middle income': 2,
        'High income': 3
    }
    df['region_economic_classification'] = df['region_economic_classification'].map(economic_map).astype('Int64')
    #print(f"Converted 'region_economic_classification' to numeric codes.")
    #print(df['region_economic_classification'].head())

    risk_columns = ['seismic_hazard_zone', 'tropical_cyclone_wind_risk', 'tornadoes_wind_risk']
    risk_map = {
        'Very Low': 0,
        'Low': 1,
        'Moderate': 2,
        'High': 3,
        'Very High': 4
    }
    for col in risk_columns:
        df[col] = df[col].map(risk_map).astype('Int64')
        #print(f"Converted column {col} to numeric.")
        #print(df[col].head())

    df['koppen_climate_zone'] = df['koppen_climate_zone'].astype('category').cat.codes.astype('Int64')

    nonnumeric_columns = df.select_dtypes(exclude=['number']).columns
    categorical_columns = df.columns.difference(numeric_columns).difference(nonnumeric_columns)
    print(f"Nonnumeric columns: {nonnumeric_columns.tolist()}")
    print(f"Categorical columns: {categorical_columns.tolist()}")

    print(df.columns)
    return df

train_tabular = convert_to_nummeric(train_tabular)
eval_tabular = convert_to_nummeric(eval_tabular)

Numeric columns: ['year', 'deflated_gdp_usd', 'us_cpi', 'straight_distance_to_capital_km', 'construction_cost_per_m2_usd']
Nonnumeric columns before processing: ['data_id', 'geolocation_name', 'quarter_label', 'country', 'developed_country', 'landlocked', 'region_economic_classification', 'access_to_airport', 'access_to_port', 'access_to_highway', 'access_to_railway', 'seismic_hazard_zone', 'flood_risk_class', 'tropical_cyclone_wind_risk', 'tornadoes_wind_risk', 'koppen_climate_zone', 'sentinel2_tiff_file_name', 'viirs_tiff_file_name']
Nonnumeric columns: ['data_id', 'sentinel2_tiff_file_name', 'viirs_tiff_file_name']
Categorical columns: ['access_to_airport', 'access_to_highway', 'access_to_port', 'access_to_railway', 'country', 'developed_country', 'flood_risk_class', 'geolocation_name', 'koppen_climate_zone', 'landlocked', 'quarter_label', 'region_economic_classification', 'seismic_hazard_zone', 'tornadoes_wind_risk', 'tropical_cyclone_wind_risk']
Index(['data_id', 'geolocation_name

Image processing

In [6]:
def load_tiff(path):
        img = tiff.imread(path)
        t = torch.tensor(img, dtype=torch.float32)
        if t.dim() == 2:
            t = t.unsqueeze(0)
        elif t.dim() == 3:
            t = t.permute(2, 0, 1)
        t = torch.nan_to_num(t, nan=0.0)
        t = TF.resize(t, [224, 224])
        return t

def normalize_image(t):
    mean = t.mean(dim=(1, 2), keepdim=True)
    std = t.std(dim=(1, 2), keepdim=True).clamp(min=1e-6)
    return (t - mean) / std

def process_image(sentinel_path, viirs_path):
    sentinel_img = load_tiff(sentinel_path)
    viirs_img = load_tiff(viirs_path)

    sentinel_img = normalize_image(sentinel_img)
    viirs_img = normalize_image(viirs_img)

    return {'sentinel': sentinel_img, 'viirs': viirs_img}

def process_images_in_df(df, imgPath):
    counter = 0
    images = df[['sentinel2_tiff_file_name', 'viirs_tiff_file_name']].itertuples(index=False)
    len_images = len(df)
    processed_files = []

    for sentinel, viirs in images:
        counter += 1
        print(f"Processing image {counter} of {len_images}", end='\r')

        sentinelFileName = "_".join(sentinel.split('.')[0].split('_')[2:])
        viirsFileName = "_".join(viirs.split('.')[0].split('_')[1:])
        
        if sentinelFileName != viirsFileName: raise ValueError(f"{sentinelFileName} and {viirsFileName} do not match")
        
        newFileName = sentinelFileName + ".pt"
        tensor = process_image(imgPath / sentinel, imgPath / viirs)
        torch.save(tensor, SavePath / "processed_composite" / newFileName)
        processed_files.append(newFileName)

    df['processed_imgs'] = processed_files

process_images_in_df(train_tabular, TrainingImgPath)
process_images_in_df(eval_tabular, ValidationImgPath)

Since the data is split into two countries Japan and Philippines, the data is split to train two separate models. For each country to enhance model performance. And since the data differs alot, this is done before nomalizing

In [7]:
# Split the data based on country
def split_df_by_country(df):
    philippines = pd.DataFrame()
    japan = pd.DataFrame()

    for row in train_tabular.itertuples(index=False):
        if row.country == "Japan" or row.country == 1:
            japan = pd.concat([japan, pd.DataFrame(data=[row])], ignore_index=True)
        elif row.country == "Philippines" or row.country == 0:
            philippines = pd.concat([philippines, pd.DataFrame(data=[row])], ignore_index=True)
        else:
            raise ValueError(f"Unknown country: {row.country}")
        
    return philippines, japan

philippines_train, japan_train = split_df_by_country(train_tabular)

print("Training Data")
print(f"All data shape: {train_tabular.shape}")
print(f"Philippines shape: {philippines_train.shape}")
print(f"Japan shape: {japan_train.shape}")
print("Eval Data")
print(f"All data shape: {eval_tabular.shape}")

Training Data
All data shape: (1024, 24)
Philippines shape: (457, 24)
Japan shape: (567, 24)
Eval Data
All data shape: (1024, 23)


Normalizing data

In [8]:
def normalize(df : pd.DataFrame, normalizing_cols  : list) -> pd.DataFrame:

    for col in normalizing_cols:
        min_val = df[col].min()
        max_val = df[col].max()
        if max_val - min_val > 0:
            df[col] = (df[col] - min_val) / (max_val - min_val)
        else:
            df[col] = 0.0
    return df

"""Normalizing the data"""
normalizing_cols = ['quarter_label', 'deflated_gdp_usd', 'us_cpi', 'straight_distance_to_capital_km']
philippines_train = normalize(philippines_train, normalizing_cols)
japan_train = normalize(japan_train, normalizing_cols)
train_tabular = normalize(train_tabular, normalizing_cols)
eval_tabular = normalize(eval_tabular, normalizing_cols)


Saving the data after preprocessing to use for model training.

In [9]:
#Drop colums with only one unique value
def drop_constant_columns(df):
    for col in df.columns:
        if df[col].nunique() == 1:
            df = df.drop(columns=[col])
    return df

def drop_redundant_columns(df):
    redundant = ['sentinel2_tiff_file_name', 'viirs_tiff_file_name', 'year']
    for col in redundant:
        if col in df.columns:
            df = df.drop(columns=[col])
    return df

def remove_matching_columns(df):
    exiting_cols = []
    for col in df:
        if col in exiting_cols:
            continue
        for other_col in df:
            if col == other_col:
                continue
            if (df[col] == df[other_col]).all():
                print(f"Column: {col}, matches column: {other_col}")
                exiting_cols.append(other_col)
    df = df.drop(columns=exiting_cols)
    return df

def process(df):
    df = drop_constant_columns(df)
    df = drop_redundant_columns(df)
    df = remove_matching_columns(df)
    return df

train_tabular = process(train_tabular)
philippines_train = process(philippines_train)
japan_train = process(japan_train)
eval_tabular = process(eval_tabular)

print("Training Data")
print(f"All data shape: {train_tabular.shape}")
print(f"Philippines shape: {philippines_train.shape}")
print(f"Japan shape: {japan_train.shape}")
print("Eval Data")
print(f"All data shape: {eval_tabular.shape}")

train_tabular.to_csv(SavePath / "processed_data.csv", index=False)
philippines_train.to_csv(SavePath / "processed_philippines.csv", index=False)
japan_train.to_csv(SavePath / "processed_japan.csv", index=False)
eval_tabular.to_csv(SavePath / "processed_eval.csv", index=False)

Column: country, matches column: developed_country
Column: country, matches column: developed_country
Training Data
All data shape: (1024, 20)
Philippines shape: (457, 19)
Japan shape: (567, 17)
Eval Data
All data shape: (1024, 19)
